In [1]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# ===============================
# CONFIG
# ===============================
INPUT_FILE = "/kaggle/input/qa-data1/qa_data.json"
OUTPUT_FILE = "/kaggle/working/qa_data_summarized_pegasus.json"

MODEL_NAME = "google/pegasus-arxiv"
MAX_INPUT_LEN = 1024
MAX_SUMMARY_LEN = 180

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ===============================
# LOAD MODEL
# ===============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# ===============================
# QUESTION-AWARE SUMMARIZATION
# ===============================
def summarize_knowledge_with_question(knowledge, question):
    text = (
        f"Question: {question}\n\n"
        f"Knowledge: {knowledge}"
    )

    inputs = tokenizer(
        text,
        truncation=True,
        max_length=MAX_INPUT_LEN,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=MAX_SUMMARY_LEN,
            num_beams=8,
            length_penalty=1.5
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# ===============================
# LOAD DATA (JSON / JSONL)
# ===============================
def load_json_or_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        first_char = f.read(1)
        f.seek(0)
        if first_char == "[":
            data = json.load(f)
        else:
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))
    return data

data = load_json_or_jsonl(INPUT_FILE)
print(f"Loaded {len(data)} samples")

# ===============================
# PROCESS + SAVE
# ===============================
new_data = []

for sample in tqdm(data, desc="Summarizing with PEGASUS"):
    summarized_knowledge = summarize_knowledge_with_question(
        sample["knowledge"],
        sample["question"]
    )

    new_data.append({
        "knowledge": summarized_knowledge,
        "question": sample["question"],
        "right_answer": sample["right_answer"],
        "hallucinated_answer": sample["hallucinated_answer"]
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(new_data, f, indent=2, ensure_ascii=False)

print(f"\n✅ File saved at: {OUTPUT_FILE}")

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

2026-02-01 07:54:22.352543: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769932462.528757      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769932462.577691      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769932462.990763      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769932462.990804      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769932462.990807      55 computation_placer.cc:177] computation placer alr

...

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-arxiv and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Loaded 10000 samples


Summarizing with PEGASUS: 100%|██████████| 10000/10000 [8:08:40<00:00,  2.93s/it] 


✅ File saved at: /kaggle/working/qa_data_summarized_pegasus.json
